# ⚡ Aula 21 — Implementação do Modelo no Projeto

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

O hackathon entrou na **fase crítica de desenvolvimento**. Os times têm o plano definido, o repositório estruturado e o baseline medido. Agora precisam implementar o modelo profundo com **aceleração GPU real**: escolher a precisão certa, otimizar o pipeline de dados, garantir que o modelo cabe na VRAM e monitorar cada experimento. O objetivo é entregar um modelo treinando corretamente, com pelo menos uma rodada completa registrada.

Este notebook roda **sem GPU** (modo de referência). Os códigos de CUDA ficam protegidos por verificação, exibindo os conceitos quando não há placa NVIDIA.

In [ ]:
# 1. Detectar GPU e PyTorch
try:
    import torch
    TEM_TORCH = True
    GPU = torch.cuda.is_available()
    AMP_OK = (hasattr(torch, "amp") and GPU)
except ImportError:
    TEM_TORCH = False
    GPU = False
    AMP_OK = False

print(f"PyTorch: {TEM_TORCH} | GPU: {GPU} | AMP disponível: {AMP_OK}")
if GPU:
    print(f"  -> {torch.cuda.get_device_name(0)}")
else:
    print("  -> Sem GPU: seguimos em MODO DE REFERÊNCIA (conceitos + números esperados).")

## 1. Pipeline de treinamento otimizado

Os quatro conceitos que compõem o pipeline: DataLoader otimizado, mixed precision, `set_to_none` e transferência `non_blocking`.

In [ ]:
# 2. Função de época com mixed precision (roda em CUDA ou em modo de referência)
def treinar_epoca_referencia(usar_amp=True):
    """Demonstra o fluxo de uma época otimizada.

    Sem PyTorch/GPU, apenas imprime os passos do pipeline para fins didáticos.
    """
    passos = [
        "DataLoader(num_workers=4, pin_memory=True, prefetch_factor=2)",
        "X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)",
        "optimizer.zero_grad(set_to_none=True)",
        "with autocast(): logits = model(X); loss = criterion(logits, y)",
        "scaler.scale(loss).backward()",
        "scaler.unscale_(optimizer); clip_grad_norm_(..., 1.0)",
        "scaler.step(optimizer); scaler.update()",
    ]
    if not usar_amp:
        passos.remove("with autocast(): logits = model(X); loss = criterion(logits, y)")
        passos.remove("scaler.scale(loss).backward()")
        passos.remove("scaler.unscale_(optimizer); clip_grad_norm_(..., 1.0)")
        passos.remove("scaler.step(optimizer); scaler.update()")
        passos += ["logits = model(X); loss = criterion(logits, y)", "loss.backward()", "optimizer.step()"]

    modo = "AMP (fp16)" if usar_amp else "FP32"
    print(f"=== Pipeline {modo} ===")
    for i, p in enumerate(passos, 1):
        print(f"  {i}. {p}")

treinar_epoca_referencia(usar_amp=True)

## 2. Benchmark fp32 vs fp16 (modelo simulado)

Como não há GPU no laboratório, usamos **números de referência** para T4/V100/RTX para prever o ganho típico do AMP.

In [ ]:
# 3. Tabela de referência do ganho de mixed precision por GPU
REFERENCIA_AMP = {
    "RTX 3090": {"speedup": 2.1, "vram_reducao": 0.45},
    "RTX 4090": {"speedup": 2.3, "vram_reducao": 0.45},
    "T4 (Colab)": {"speedup": 1.7, "vram_reducao": 0.40},
    "V100 (Colab Pro)": {"speedup": 2.0, "vram_reducao": 0.45},
    "A100 (bf16)": {"speedup": 2.5, "vram_reducao": 0.50},
}

IMGS_S_FP32 = 180.0   # throughput de referência em fp32
print(f"Throughput fp32 (base): {IMGS_S_FP32:.0f} imgs/s\n")
print(f"{'GPU':<18} {'fp16 (imgs/s)':>14} {'redução VRAM':>14}")
print("-" * 50)
for gpu, r in REFERENCIA_AMP.items():
    fp16 = IMGS_S_FP32 * r["speedup"]
    print(f"{gpu:<18} {fp16:>14.0f} {r['vram_reducao']*100:>13.0f}%")

## 3. Estimar batch efetivo com gradient accumulation

Quando o batch ideal não cabe na VRAM, acumule gradientes por N steps.

In [ ]:
# 4. Calcular os steps de acumulação necessários
def calcular_acumulacao(batch_real, batch_efetivo_desejado):
    """Retorna quantos steps de acumulação usar (arredonda para baixo)."""
    if batch_real <= 0:
        return None
    return max(1, batch_efetivo_desejado // batch_real)

BATCH_REAL = 32
for alvo in (64, 128, 256):
    n = calcular_acumulacao(BATCH_REAL, alvo)
    print(f"batch_real={BATCH_REAL} -> batch_efetivo={alvo}: {n} steps de acumulação")

## 4. Estimar a VRAM combinando técnicas

Cada técnica reduz o consumo de VRAM. O script abaixo estima quanto sobra ao combiná-las.

In [ ]:
# 5. Estimar VRAM com/sem mixed precision
def estimar_vram_mb(batch_size, por_amostra=100.0, amp=False, overhead=1500):
    """Estimativa de VRAM: overhead do runtime + ativações por amostra."""
    fator = 0.5 if amp else 1.0
    return overhead + batch_size * por_amostra * fator

VRAM_GPU = 24576  # RTX 4090 (MB)
print(f"VRAM da GPU (RTX 4090): {VRAM_GPU} MB\n")
for bs in (16, 32, 64):
    fp32 = estimar_vram_mb(bs, amp=False)
    fp16 = estimar_vram_mb(bs, amp=True)
    cabe = "✅ cabe" if fp16 < VRAM_GPU else "❌ OOM"
    print(f"batch={bs:<3} | fp32={fp32:>6.0f} MB | fp16={fp16:>6.0f} MB | {cabe}")

## 5. Simular o profiling (top operações)

O `torch.profiler` ordena operações por tempo na GPU. Abaixo, uma simulação do relatório típico de um ResNet.

In [ ]:
# 6. Simulação do relatório do profiler (top operações por tempo CUDA)
TOP_OPERACOES = [
    ("cudnn_convolution_backward", 38.2, "backward"),
    ("cudnn_convolution", 27.5, "forward"),
    ("ampere_sgemm_128x64", 14.1, "forward"),
    ("elementwise_kernel", 7.8, "optimizer"),
    ("batch_norm", 4.2, "forward"),
]
tempo_total = sum(t for _, t, _ in TOP_OPERACOES)
print(f"{'Operação':<32} {'ms':>7} {'%':>6}  etapa")
print("-" * 62)
for nome, ms, etapa in TOP_OPERACOES:
    print(f"{nome:<32} {ms:>7.1f} {100*ms/tempo_total:>5.0f}%  {etapa}")
print(f"\nOnde otimizar: as convoluções (backward+forward) dominam — AMP ataca justamente elas.")

---
## 📝 6. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Comparar pipeline fp32 vs fp16

Chame `treinar_epoca_referencia(usar_amp=False)` e `treinar_epoca_referencia(usar_amp=True)` e liste as 3 principais diferenças de fluxo.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# treinar_epoca_referencia(False); treinar_epoca_referencia(True)

### Exercício 2: Calcular acumulação para um alvo

Se o batch real é **24** e vocês querem batch efetivo **96**, quantos steps de acumulação são necessários? Use `calcular_acumulacao`.

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# print(calcular_acumulacao(24, 96))

### Exercício 3: Batch máximo com AMP

Usando `estimar_vram_mb` com AMP, encontre o **maior batch** (múltiplo de 8) que cabe em 24 GB.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# for bs in range(8, 129, 8): if estimar_vram_mb(bs, amp=True) < 24576: ultimo = bs

### Exercício 4: Identificar o gargalo

A partir de `TOP_OPERACOES`, calcule qual **percentual** do tempo está no *backward* e qual técnica (AMP, checkpointing, DataLoader) ataca esse gargalo.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# pct_bw = sum(t for _,t,e in TOP_OPERACOES if e=='backward') / tempo_total

### Exercício 5: Estimar o speedup total do projeto

Combine o ganho do AMP (speedup da GPU escolhida) com um DataLoader otimizado (1.3×). Estime o throughput final sobre o baseline fp32 de 180 imgs/s e imprima a frase de resumo.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# speedup_amp = REFERENCIA_AMP['RTX 4090']['speedup']; total = 180*speedup_amp*1.3

---
## 📌 7. Síntese e Conclusão

- **autocast + GradScaler:** o par que habilita mixed precision — sempre usar em RTX/A100/H100.
- **pin_memory=True:** fixa os dados na RAM do host para transferência assíncrona rápida.
- **Gradient checkpointing:** recomputa ativações no backward — −60% VRAM, +30% tempo.
- **Gradient accumulation:** batch efetivo = batch real × n_steps, sem OOM.
- **torch.profiler:** meça antes de otimizar — evita otimizar a parte errada do pipeline.
- **Logging contínuo:** registrar VRAM e throughput por época detecta regressões cedo.

**Tarefa para a Aula 22:** pelo menos 10 épocas completas com AMP e logging, e o `monitor_gpu.sh` salvo em CSV.